**Why ingestion exists at all** comes down to one limitation: a model only knows what's in its training data and whatever you put in the prompt — never your internal documents, never today's FD rates, never an SOP written last week.
- This is the exact same gap RAG exists to close — ingestion is just the **first** stage of that pipeline, the part that gets raw files into a shape anything downstream can use
- The "new-product-name problem" is really an ingestion-and-retrieval problem stacked together: the knowledge exists somewhere (a PDF), it's just never been **ingested** into anything the model can search
- Get ingestion wrong, and every later stage — chunking, embedding, retrieval — inherits the damage. Garbage in, garbage out applies here more than almost anywhere else in the pipeline

**The Document object pattern** is the one decision that makes everything downstream source-agnostic.
- Every loader — PDF, text file, CSV — returns the same shape: `page_content` (the actual text) plus `metadata` (a dict describing where it came from)
- Once a `Document` exists, chunking and embedding code never needs an `if it's a PDF / if it's a CSV` branch anywhere — that's the entire point of standardizing on one shape
- `knowledge_base's `{"text":, "source":}` dicts were a stripped-down version of this same idea — this sub-chapter formalizes it into a real `@dataclass`, with richer metadata (page numbers, row indices) that the dict version was throwing away

**PDF loading** has a clean path and a fallback path, and conflating them is a common mistake.
- `PDFLoader` mirrors LangChain's `PyPDFLoader` — one `Document` per page, not one per file, since that's the natural retrieval unit for a multi-page policy doc
- **`pdfplumber`** is the upgrade when a PDF has real tables (interest-rate tables, tenure charts) — `pypdf`'s `extract_text()` flattens tables into messy single-line text; `pdfplumber`'s `extract_tables()` keeps row/column structure intact
- **Scanned PDFs have no text layer at all** — `pypdf` silently returns empty strings, not an error, which is the dangerous part: a scanned SOP fails *quietly*, looking like an empty document instead of throwing something you'd notice
- `is_scanned_pdf()` checks **every** page, not just the first — a real SOP could have a typed cover page followed by scanned signature pages
- `load_pdf_smart()` encodes the actual production rule: try the real text layer first, only fall back to OCR (Tesseract) if the PDF turns out to be scanned. Never OCR text that was never an image — it's strictly slower and lower quality than text extraction

**Text loading** is the simplest loader here, on purpose — `fd_keyword_groups.txt` and `fd_negation_phrases.txt` are small, structured reference files, not documents that benefit from being split into multiple `Document`s.
- `TextLoader` reads the whole file into **one** `Document` — splitting these into pieces would actively hurt: a keyword group only makes sense as a complete list, not torn into fragments
- This is the same loader that would handle plain-text SOPs or logs if the NBFC ever has them — the loader doesn't care about content, only about "give me everything in this file as one block"

**Why drop PDF handling entirely, rather than keep it as a fallback?** — this is a separation-of-concerns decision, not just "less code to maintain."
- Parsing PDFs (tables, scanned pages, OCR, encrypted files) is a genuinely hard, specialized problem — hard enough that an entire team exists just to solve it well, once, centrally
- If this pipeline kept its own `PDFLoader` *and* consumed the conversion team's JSON, you'd have **two independent PDF-parsing code paths** that could silently disagree with each other — same source PDF, two different extracted texts, depending on which path happened to touch it
- Removing it isn't "trusting blindly" — it's **picking one source of truth** for "what does this PDF actually say," and putting the validation effort into checking *that* source, rather than maintaining a second, weaker parser as a hedge

**The Document object pattern just proved its actual value** — this is the payoff of a decision made all the way back at the start of Sub-Chapter 1.
- Every loader, regardless of source, has always returned the same shape: `{"page_content":, "metadata":}`
- Because of that, swapping the *entire upstream source* — PDFs in, JSON in — required changing **exactly one file**, and nothing downstream (chunking, embedding, search, dedup) needed to know anything changed at all
- If chunking code had been written assuming "documents come from PDFs" anywhere, this swap would have meant hunting down and fixing every place that assumption leaked in. It didn't, because the abstraction was honest from the start
- This is the concrete answer to "why bother standardizing a shape early" — not a hypothetical, you just watched it pay off

**`JSONLoader` is doing the same *job* `PDFLoader` did, just reading a different *source*.**
- Same granularity: one `Document` per page, not one per file — that choice didn't change just because the source format did
- Same metadata intent, richer in practice: `document_code`, `version`, `ocr_confidence`, `was_scanned` all ride along now, because the conversion team's JSON happens to carry information that was expensive or impossible for us to reconstruct from a raw PDF ourselves
- This is worth noticing as a general pattern: a well-designed upstream format can hand you metadata *for free* that you'd have had to build dedicated detection logic for otherwise (`is_scanned_pdf()` doesn't exist anymore — we just read `was_scanned` off the JSON instead of re-deriving it)

**"Trust but verify" is a specific design stance, not a vague caution.**
- Removing our own PDF parser means we **lost the ability to independently check** whether the conversion team's JSON is actually correct — we can no longer go back to the original PDF ourselves and compare
- The honest response to losing that check isn't "trust them completely" — it's **moving the check to a different point**: validate the JSON's *shape and plausibility* on the way in, even though we can't validate its *accuracy* against the source anymore
- `validate_ocr_json()` checks exactly two cheap, generic signals — suspiciously short text, and low OCR confidence — that don't require knowing anything about FD content specifically. It can't tell you the text is *correct*, only that it's *plausible enough to be worth ingesting*
- This is also why it **warns instead of raising by default** — a Lead-level judgment call, not an oversight: blocking ingestion entirely on every warning would mean one bad page in a 50-page document throws away the other 49 good ones. Logging and continuing, with the option to raise when you want stricter behavior, is the more production-realistic default

**What deliberately did NOT change, and why that's the interesting part.**
- `TextLoader`, `CSVLoader`, `find_duplicates()`, the incremental manifest — none of these ever had a PDF dependency buried inside them, so none of them needed to change
- That's not luck — it's what happens when a pipeline's stages are genuinely decoupled. The "ingestion" stage changed its *input format*; the "what do we do with a Document once we have one" stages never knew or cared
- The one thing that *did* shift inside an otherwise-unchanged function: the incremental manifest now hashes `.json` files instead of `.pdf` files. Same function, same logic, just pointed at a different file extension — proof the manifest was never actually PDF-specific, just file-path-and-hash-specific all along

**The honest trade-off, restated now that it's real code and not just a discussion.**
- **Gained**: this pipeline is now simpler, has fewer dependencies (no `pytesseract`, no `pdf2image`, no OCR logic to maintain), and benefits from a team that's *better* at PDF parsing than a one-off loader would ever be
- **Gave up**: independent verification against the source PDF, and a new dependency on someone else's pipeline being timely and correct
- **The new risk that's now load-bearing**: `validate_ocr_json()` is the *only* defense against a bad conversion reaching the knowledge base. If that function's two checks (length, confidence) aren't enough to catch a real failure mode — say, text that's long and "confident" but actually OCR'd the wrong page, or swapped two pages' order — there's currently nothing else standing between a silent conversion error and a customer-facing wrong answer

**CSV/Excel loading** treats every row as its own retrievable unit, which is a different granularity decision than the text loader made.
- `CSVLoader` mirrors LangChain's `CSVLoader`: **one `Document` per row**, with every column folded into `page_content` as `key: value` lines, and the row index kept in `metadata`
- Run against the real `fd_master_database.csv`, row 0 becomes a `Document` whose content is Shobha Chopra's entire FD record — that's now something you could chunk, embed, and search individually, not just `pandas`-filter
- This is the loader that would matter if the agent ever needs to **search across customer records semantically** rather than by exact `FD_No` lookup — which is exactly the access-control question flagged for Sub-Chapter 5

**`load()` vs `lazy_load()`** is a memory decision that's invisible at your current scale and very visible at the NBFC's real scale.
- `load()` is, quite literally, `list(self.lazy_load())` in every loader above — it isn't a separate implementation, it's lazy loading with the patience removed
- At 1 PDF or 20 CSV rows, the difference is unmeasurable. At 50,000 real PDFs, `load()` tries to hold every page of every document in memory **simultaneously** before you've even started chunking — that's the actual mechanism behind a memory crash, not an abstract warning
- `lazy_load()` yields one `Document` at a time — you can stop early if you found what you needed, and memory usage stays flat regardless of how many documents exist

**Duplicate documents under different filenames** — `FD_Policy.pdf` / `FD_Policy_Final.pdf` / `FD_Policy_V2.pdf` — is a two-stage problem, cheapest check first.
- **Stage 1, hash**: `content_hash()` (SHA-256) catches **exact** duplicates instantly — same content, byte for byte, regardless of filename. Computing a hash is nearly free, so this runs on everything
- **Stage 2, cosine similarity**: only computed for documents that **survived** the hash check — embeddings are the expensive step, so paying for them on content you'd already have rejected is wasted cost
- Tested against 4 documents simulating exactly this filename pattern: the byte-identical pair was caught by the hash, and a *reworded* near-duplicate (same fact, different phrasing) was caught by cosine similarity — two different kinds of duplication, two different tools, applied in cost order

**Incremental ingestion** is the same discipline `insert_fd_record()` / `update_fd_record()` already established for the FD database, applied to documents instead of rows.
- A **manifest** (`ingestion_manifest.json`) tracks each source file's content hash from the last time it was ingested
- `get_files_to_ingest()` compares current file hashes against the manifest and returns **only** what's new or changed — everything else gets skipped entirely, no re-chunking, no re-embedding, no wasted API calls
- Tested directly: changed the content of exactly one tracked file, and confirmed only **that** file showed up for re-ingestion — the unchanged file never got touched, which is the entire point and the part most naive "just reload everything" pipelines get wrong

In [16]:
import os
import csv
import json
import hashlib
from datetime import datetime, timezone
import glob
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
 
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(EMBED_MODEL_NAME)

In [5]:
JSON_FOLDER = "../data/related_documents_json/"
 
 
def make_document(page_content: str, metadata: dict = None) -> dict:
    # A "Document" is just a dict: the text itself, plus a note on where it came from.
    return {"page_content": page_content, "metadata": metadata or {}}

# ----------------------------------------------------------------------
# JSON loading -- reads the conversion team's output, one Document per page.
# ----------------------------------------------------------------------
 
MIN_TEXT_CHARS = 20          # below this many characters, treat a page as "basically empty"

In [6]:

def validate_ocr_json(data: dict, file_path: str) -> list:
    # Checks one JSON file's contents for anything suspicious BEFORE we use it.
    # Returns a list of warning messages (empty list = nothing wrong found).
    # NOTE: we do NOT re-check ocr_confidence here -- the conversion/OCR team
    # owns that quality bar on their side. We still store ocr_confidence in
    # metadata (see below) so it's visible if anyone wants to look at it later.
    warnings = []
 
    if "pages" not in data or not data["pages"]:
        # no pages at all -- nothing further to check, bail out early
        warnings.append(f"{file_path}: no 'pages' key, or pages list is empty")
        return warnings
 
    for page in data["pages"]:
        page_num = page.get("page_number", "?")
        text = page.get("text", "")
 
        if len(text.strip()) < MIN_TEXT_CHARS:
            # very little/no text -- likely a page that failed to convert properly
            warnings.append(
                f"{file_path} page {page_num}: text is suspiciously short "
                f"({len(text.strip())} chars) -- likely a failed conversion"
            )
 
    return warnings

In [7]:
def lazy_load_json(file_path: str, raise_on_warning: bool = False):
    # Reads ONE json file and gives back Documents one page at a time (a generator).
    with open(file_path, encoding="utf-8") as f:
        data = json.load(f)  # parse the JSON file into a Python dict
 
    warnings = validate_ocr_json(data, file_path)  # run the checks above
    for w in warnings:
        print(f"  [WARNING] {w}")  # always print warnings, even if we don't stop
    if warnings and raise_on_warning:
        # only actually STOP if the caller explicitly asked us to be strict
        raise ValueError(f"{file_path} failed validation: {warnings[0]}")
 
    for page in data.get("pages", []):
        # turn each page in the JSON into one Document dict
        yield make_document(
            page_content=page.get("text", ""),
            metadata={
                "source": data.get("source_pdf", file_path),
                "document_code": data.get("document_code"),
                "version": data.get("version"),
                "page": page.get("page_number"),
                "ocr_confidence": page.get("ocr_confidence"),
                "was_scanned": page.get("was_scanned"),
            },
        )

In [8]:
def load_json(file_path: str, raise_on_warning: bool = False) -> list:
    # Same as lazy_load_json, but returns a normal list instead of a generator.
    return list(lazy_load_json(file_path, raise_on_warning=raise_on_warning))
 
 
def load_all_json(folder: str = JSON_FOLDER) -> list:
    # Loads EVERY .json file in the given folder, one after another.
    documents = []
    for path in sorted(glob.glob(os.path.join(folder, "*.json"))):  # find all .json files in the folder
        documents.extend(load_json(path))
    return documents

In [9]:
# ----------------------------------------------------------------------
# Text loading -- fd_keyword_groups.txt / fd_negation_phrases.txt.
# Whole file becomes ONE Document (not split into pieces).
# ----------------------------------------------------------------------
 
def lazy_load_text(file_path: str):
    # Reads the whole file in one go and gives back exactly ONE Document.
    with open(file_path, encoding="utf-8") as f:
        yield make_document(page_content=f.read(), metadata={"source": file_path})
 
 
def load_text(file_path: str) -> list:
    return list(lazy_load_text(file_path))

In [10]:
# ----------------------------------------------------------------------
# CSV loading -- fd_dataset_messy.csv, fd_master_database.csv.
# Each ROW becomes its own separate Document.
# ----------------------------------------------------------------------
 
def lazy_load_csv(file_path: str):
    # Reads the CSV row by row, turning each row into its own Document.
    with open(file_path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)  # reads each row as a dict: {column_name: value}
        for i, row in enumerate(reader):
            # join all columns into one block of text, e.g. "FD_No: BJ2019FD7717\nCustomer_Name: ..."
            content = "\n".join(f"{k}: {v}" for k, v in row.items())
            yield make_document(
                page_content=content,
                metadata={"source": file_path, "row": i},
            )
 
def load_csv(file_path: str) -> list:
    return list(lazy_load_csv(file_path))

In [11]:
# ----------------------------------------------------------------------
# Production issue: duplicate documents under different filenames.
# Step 1: hash check (catches EXACT duplicates, cheap).
# Step 2: cosine similarity (catches NEAR duplicates, only on what's left).
# ----------------------------------------------------------------------
 
def content_hash(text: str) -> str:
    # A fingerprint for a piece of text -- identical text always gives the identical hash.
    return hashlib.sha256(text.encode("utf-8")).hexdigest()
 
 
def find_duplicates(documents: list, near_duplicate_threshold: float = 0.97) -> list:
    # Returns a list of (this_doc_index, duplicate_of_this_index, "exact" or "near").
    seen_hashes = {}        # hash -> index of the FIRST document we saw with that hash
    kept_for_embedding = [] # (index, text) for docs that were NOT an exact duplicate
    duplicates = []
 
    for i, doc in enumerate(documents):
        h = content_hash(doc["page_content"])
        if h in seen_hashes:
            # exact same text as something we already saw -- exact duplicate, stop here
            duplicates.append((i, seen_hashes[h], "exact"))
        else:
            # new content -- remember its hash, and keep it for the slower check below
            seen_hashes[h] = i
            kept_for_embedding.append((i, doc["page_content"]))
 
    if len(kept_for_embedding) > 1:
        # only NOW do we pay for embeddings -- and only on documents that survived the hash check
        indices, texts = zip(*kept_for_embedding)
        embeddings = model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)
        already_flagged = set()  # don't double-report a document that's already a known duplicate
        for a in range(len(indices)):
            if indices[a] in already_flagged:
                continue
            for b in range(a + 1, len(indices)):
                if indices[b] in already_flagged:
                    continue
                score = float(np.dot(embeddings[a], embeddings[b]))  # cosine similarity (vectors are normalized)
                if score >= near_duplicate_threshold:
                    duplicates.append((indices[b], indices[a], f"near ({score:.3f})"))
                    already_flagged.add(indices[b])
 
    return duplicates

In [12]:
# ----------------------------------------------------------------------
# Incremental ingestion -- skip files that haven't changed since last run.
# ----------------------------------------------------------------------
 
MANIFEST_PATH = "ingestion_manifest.json"  # where we remember what we've already ingested
 
 
def load_manifest(path: str = MANIFEST_PATH) -> dict:
    # Reads our "memory" of what's already been ingested. Empty dict if we've never run before.
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}
 
 
def save_manifest(manifest: dict, path: str = MANIFEST_PATH) -> None:
    # Writes our "memory" back to disk so the NEXT run can read it.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
 
 
def get_files_to_ingest(file_paths: list, manifest: dict) -> list:
    # Compares each file's CURRENT hash against the hash we saved last time.
    # Different hash (or never seen before) = needs to be (re)ingested.
    to_ingest = []
    for path in file_paths:
        with open(path, "rb") as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()
        if manifest.get(path, {}).get("hash") != file_hash:
            to_ingest.append(path)
    return to_ingest
 
 
def update_manifest_entry(manifest: dict, file_path: str) -> None:
    # Records "this file, with this exact hash, was ingested just now."
    with open(file_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    manifest[file_path] = {
        "hash": file_hash,
        "last_ingested": datetime.now(timezone.utc).isoformat(),
    }


In [13]:
docs = load_json("../data/related_documents_json/02_FD_Product_Guide.json")
print(f"Loaded {len(docs)} page(s)")
print(f"  page_content[:80] : {docs[0]['page_content'][:80]}...")
print(f"  metadata          : {docs[0]['metadata']}")

Loaded 3 page(s)
  page_content[:80] : Fixed Deposit Product Guide
Page 1
Internal Reference – Customer Service
 Fixed ...
  metadata          : {'source': '02_FD_Product_Guide.pdf', 'document_code': 'FD-PROD-02', 'version': '1.0', 'page': 1, 'ocr_confidence': None, 'was_scanned': False}


In [17]:
# ----- Demo 3: Loading the whole JSON folder at once -----
all_docs = load_all_json()
print(f"Loaded {len(all_docs)} total pages across all 4 documents")
sources = sorted(set(d["metadata"]["source"] for d in all_docs))
print(f"  Sources: {sources}")

Loaded 17 total pages across all 4 documents
  Sources: ['01_FD_FAQ.pdf', '02_FD_Product_Guide.pdf', '03_FD_SOPs.pdf', '04_FD_Policy_Reference.pdf']


In [19]:
# ----- Demo 4: Text loading -----
text_docs = load_text("../data/fd_keyword_groups.txt")
print(f"Loaded {len(text_docs)} document (whole file as one chunk)")
print(f"  metadata : {text_docs[0]['metadata']}")

Loaded 1 document (whole file as one chunk)
  metadata : {'source': '../data/fd_keyword_groups.txt'}


In [20]:
# ----- Demo 5: CSV loading -- one Document per row -----
csv_docs = load_csv("../data/fd_master_database.csv")
print(f"Loaded {len(csv_docs)} rows as Documents")
print(f"  Document 0 metadata: {csv_docs[0]['metadata']}")

Loaded 20 rows as Documents
  Document 0 metadata: {'source': '../data/fd_master_database.csv', 'row': 0}


In [21]:
# ----- Demo 6: Duplicate detection -- exact (hash) + near (cosine) -----
dup_test_docs = [
    make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy.json"}),
    make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy_Final.json"}),
    make_document("FD rate is 7.1 percent for a 24 month tenure period.", {"source": "FD_Policy_V2.json"}),
    make_document("Premature withdrawal incurs a 1 percent penalty.", {"source": "FD_Policy.json"}),
]
dupes = find_duplicates(dup_test_docs, near_duplicate_threshold=0.85)
for idx, of_idx, kind in dupes:
    print(f"  Document {idx} ({dup_test_docs[idx]['metadata']['source']}) is a {kind} "
          f"duplicate of Document {of_idx} ({dup_test_docs[of_idx]['metadata']['source']})")

  Document 1 (FD_Policy_Final.json) is a exact duplicate of Document 0 (FD_Policy.json)
  Document 2 (FD_Policy_V2.json) is a near (0.993) duplicate of Document 0 (FD_Policy.json)


In [22]:
# ----- Demo 7a: Incremental ingestion manifest -- FIRST run -----
manifest = load_manifest()
tracked_files = sorted(glob.glob(os.path.join(JSON_FOLDER, "*.json")))
 
to_ingest = get_files_to_ingest(tracked_files, manifest)
print(f"  Files to ingest: {[os.path.basename(f) for f in to_ingest]}")
for f in to_ingest:
    update_manifest_entry(manifest, f)
save_manifest(manifest)

  Files to ingest: ['01_FD_FAQ.json', '02_FD_Product_Guide.json', '03_FD_SOPs.json', '04_FD_Policy_Reference.json']


In [23]:
# ----- Demo 7b: Incremental ingestion manifest -- SECOND run (run this AFTER 7a) -----
manifest = load_manifest()
to_ingest = get_files_to_ingest(tracked_files, manifest)
print(f"  Files to ingest: {to_ingest}  (expect empty -- nothing changed)")

  Files to ingest: []  (expect empty -- nothing changed)


**Top of the file runs first, before anything else** — this is just setup, no actual work happens yet.
- `model = SentenceTransformer(...)` loads the embedding model into memory once — this takes a few seconds the first time, then sits ready for later
- Every `class` and `def` below that — `PDFLoader`, `TextLoader`, `CSVLoader`, `find_duplicates`, etc. — is just a **definition**. Nothing inside them runs yet. Think of it like writing a recipe down — writing it doesn't cook anything, you still have to follow it
- The real action only starts at `if __name__ == "__main__":` near the bottom — that's the part that actually calls these functions, in order

**Step 1: PDF loading.** `pdf_loader.load()` runs first.
- Inside `load()`, it just calls `lazy_load()` and turns whatever comes out into a full list
- `lazy_load()` opens the PDF, goes page by page, and for each page calls `make_document()` to package the text + a `{"source":, "page":}` note into one dict
- End result: a list of dicts, one per page — printed to show the first one

**Step 2: checking if a PDF is scanned, then OCR if needed.**
- `is_scanned_pdf()` runs on two different PDFs — one real-text PDF, one scanned-style PDF — just to show it correctly says "no" for one and "yes" for the other
- Then `load_pdf_smart()` runs on the scanned one. Inside, it calls `is_scanned_pdf()` again, gets `True`, and because of that decides to use `OCRPDFLoader` instead of the normal `PDFLoader`
- `OCRPDFLoader` turns each PDF page into an image, then runs Tesseract OCR on that image to pull text out of the picture — that's the only way to get text from something that's really just a photo of words

**Step 3: text loading.** `TextLoader(...).load()` runs on `fd_keyword_groups.txt`.
- It just opens the file, reads the whole thing as one block, and wraps it in one `make_document()` call
- No looping, no splitting — the whole file becomes a single dict

**Step 4: CSV loading.** `CSVLoader(...).load()` runs on `fd_master_database.csv`.
- This one **does** loop — once per row in the CSV
- For each row, it joins every column into one block of text (`"FD_No: ... \nCustomer_Name: ..."` etc.) and wraps that in `make_document()`, with the row number saved in the metadata
- End result: 20 separate dicts, one per customer record

**Step 5: finding duplicates.** `find_duplicates()` runs on a small hand-made list of 4 test documents.
- First it loops through all 4 and hashes each one's text — if two hashes match exactly, that's an **exact** duplicate, caught immediately, no embedding needed
- For everything that **didn't** get caught by the hash, it sends all of those texts to the embedding model in one batch, then compares every pair's similarity score
- If a pair scores above the threshold, that's a **near** duplicate — different wording, basically the same meaning
- The function returns a list of "this one is a duplicate of that one" results, which then get printed

**Step 6: the incremental ingestion check.** This runs twice in a row, on purpose, to show the before/after.
- **First run**: `load_manifest()` loads an empty dict (no manifest file exists yet). `get_files_to_ingest()` checks both tracked files against that empty manifest — since neither has been seen before, both come back as "needs ingesting." Then `update_manifest_entry()` records each file's hash, and `save_manifest()` writes that to disk
- **Second run**: `load_manifest()` now loads the file just saved. `get_files_to_ingest()` checks the same two files again — same content, same hash as what's recorded — so this time it returns an **empty list**. Nothing needs re-processing
- That's the whole point made visible: re-run the script as many times as you want, and already-ingested files get skipped every time, until you actually change one

## PDF Loading & Document Ingestion

**Q: Issues faced while loading PDFs?**
A: Several real ones, not just "sometimes it doesn't work":
- **No text layer (scanned PDFs)** — `pypdf` doesn't error out, it silently returns an empty string. That's the dangerous part: it looks like an empty document, not a failure, so nothing flags it unless you explicitly check.
- **Mixed documents** — a 200-page SOP that's typed except for one scanned signature page at the end. A document-level "is this scanned?" check can miss that one bad page entirely; it needs to be checked per page.
- **Broken table structure** — `pypdf.extract_text()` flattens tables into a single messy line of text, losing row/column meaning. Interest-rate tables or tenure charts become unusable. `pdfplumber.extract_tables()` is the fix when tables matter.
- **Multi-column layouts** — text extraction can read across columns instead of down them, scrambling sentence order.
- **Encrypted/password-protected PDFs** — extraction fails outright unless you handle decryption first.
- **Corrupted files** — a truncated or malformed PDF can throw a hard exception mid-batch; a single bad file shouldn't be able to kill an entire ingestion run.
- **Memory at scale** — `load()` pulls every page of every PDF into memory at once. Fine for 1 file, a real risk at 50,000 files. `lazy_load()` (a generator) is the actual fix, not a nice-to-have.

**Q: Suppose there are 1000 documents — all different filenames, but a few contain identical or near-identical content. What's your approach?**
A: Filename is irrelevant; content is what matters, so the approach is two-stage and cost-ordered:
1. **Hash first.** Compute a SHA-256 hash of each document's content. Group documents by hash in a dict — any hash with more than one document is an **exact** duplicate set, regardless of what the files are named. This is `O(n)` and essentially free.
2. **Embed and compare only what survives.** For documents that didn't exact-match anything, generate embeddings and compute cosine similarity between them to catch **near**-duplicates — same information, reworded. This step is deliberately gated behind the hash check, because embeddings cost real compute; you don't want to pay for them on content you'd have rejected for free.
3. **Pick a similarity threshold deliberately, not by guessing.** Whatever number you land on (e.g. 0.97) should be validated against a labeled set of known duplicate/non-duplicate pairs, tuned to a target precision/recall — not asserted with confidence and never checked.

At 1000 documents this is trivial — hashing is `O(n)`, and even an `O(n²)` pairwise similarity pass on whatever's left after hashing is small enough not to matter. (That stops being true at scale — covered further down.)

---

## Deduplication & Hashing — Deeper

**Q: Why SHA-256 instead of MD5 for the hash?**
A: Two reasons, not one. Practically, MD5 has known collision weaknesses and is broadly flagged in security review, so there's no reason to choose it when SHA-256 costs essentially the same at this data volume. There's no real performance trade-off being given up here — it's a case where the "more secure" choice is also free.

**Q: Hashing is byte-exact. Give a realistic case where two documents that *should* count as duplicates hash differently.**
A: Trailing whitespace differences, a PDF re-saved with different embedded metadata/creation timestamps that bleed into the extracted text, or the same scanned page OCR'd twice producing slightly different character recognition. All three are "the same document" to a human and "completely different" to a hash function, because a hash has zero tolerance for any byte-level difference.

**Q: Our `find_duplicates()` is `O(n²)` — every surviving document compared against every other. That's invisible at 4 test documents. What breaks at 200,000 real documents, and what's the actual fix?**
A: At 200,000 documents, all-pairs comparison is computationally infeasible — that's hundreds of millions of similarity calculations, not a slow query, a non-starter. The real fix is **approximate nearest-neighbor (ANN) search**: for each new document, query an existing vector index for its nearest neighbor(s) and only compare against those candidates, not the entire corpus. This is exactly the kind of problem a real vector database's indexing (HNSW, IVF) is built to solve — which is the natural bridge into why a production system doesn't keep dedup logic as a standalone all-pairs loop at all.

**Q: Two FD policy PDFs are almost identical except the interest rate changed from 7% to 7.1%. Could near-duplicate logic flag the *updated* policy as a duplicate of the *old* one and discard it?**
A: Yes, and that's a genuine risk, not a hypothetical. High lexical and semantic overlap on a document that differs in exactly the one field that matters is precisely the case cosine similarity handles badly — the documents *are* 95%+ similar, the 5% difference just happens to be the only part anyone cares about. The fix isn't a cleverer threshold; it's a **field-aware exception list** — for document types where a single number is the entire point (rates, penalties, amounts), near-duplicate detection should never be allowed to auto-discard, only flag for review.

**Q: Dedup as built only compares documents within the same ingestion batch. What's missing for a document ingested today that duplicates something ingested six months ago?**
A: You'd need to compare against the **existing** vector store, not just the new batch — that's a different architecture entirely: query-against-index, not all-pairs-within-a-list. Batch-local dedup quietly assumes nothing before today's run matters, which is wrong the moment ingestion runs more than once.

---

## OCR & Scanned PDFs — Deeper

**Q: `is_scanned_pdf()` returns `True` only if *not even one* page has real text. Where does that logic give the wrong answer?**
A: A document that's mostly typed but has one scanned page buried inside — a signature page at the end of an otherwise-typed SOP, for example. Document-level "is this scanned" says "no" (most pages have text), so that one page silently returns empty text and nobody notices, because the document *as a whole* didn't trip the check. The real fix is checking and handling **per page**, not per document — accept that a single PDF can be a mix of both types.

**Q: OCR output quality varies. How would you actually measure it across a large ingestion run, rather than eyeballing a few outputs?**
A: Tesseract returns per-word confidence scores, not just text. Aggregate those across a document and flag anything below a threshold for human review, instead of trusting every OCR'd document equally. "We ran OCR" and "we ran OCR and it actually worked" are different claims, and only the confidence scores let you tell them apart at scale.

**Q: Where should OCR run in the pipeline — synchronously during ingestion, or as a background job?**
A: Depends entirely on the latency budget of the ingestion path. If ingestion is user-facing or blocking anything downstream, OCR — which is slow relative to native text extraction — has no business running inline. It belongs in an async job, with the document marked "pending" until OCR completes, not held up in a synchronous request.

---

## Document Object Pattern & Loader Design

**Q: We standardized every loader (PDF, text, CSV) on the same `{"page_content":, "metadata":}` shape. What does *not* doing this actually cost you?**
A: Without a shared shape, every downstream consumer — chunking, embedding, search — needs format-specific branching logic: "if this came from a PDF, do X; if from a CSV, do Y." One schema change anywhere then means touching every consumer individually. With a shared shape, you change the loader once and everything downstream keeps working unmodified.

**Q: Six months from now, you add a `"language"` field to metadata. What happens to documents already ingested under the old shape?**
A: They simply won't have that key. Code that consumes metadata needs to handle a missing key gracefully (`metadata.get("language", "unknown")`, not `metadata["language"]`) rather than assuming every document — old and new — has every field that exists today. This is a schema-evolution problem, and the fix is defensive reads, not a forced re-ingestion of everything that came before.

---

## Incremental Ingestion Manifest

**Q: The manifest is a single JSON file. Two ingestion jobs run concurrently and both try to update it. What happens?**
A: Last write wins — whichever job saves last silently overwrites the other's updates, with no error and no warning. At the NBFC's real scale, the actual fix is a proper datastore with atomic/transactional writes (even something as simple as a SQLite table with a write lock), not a flat JSON file being read-modified-written by multiple processes.

**Q: The manifest keys on file path, not content. A file gets renamed but the content is byte-identical. What happens, and is that the right behavior?**
A: It gets treated as brand new and fully re-ingested — a false positive, wasted compute on content that hasn't actually changed. Whether that's "wrong" is a genuine trade-off: fixing it means also hashing against the manifest's existing content hashes, not just the path, which adds complexity for a case that may be rare in practice. Senior-level answer isn't "obviously fix it" — it's naming the trade-off and deciding based on how often renames actually happen in your real source.

**Q: If ingestion crashes halfway through processing 10 changed files, what state is the manifest left in, and does a restart resume correctly?**
A: Trace the actual code: `update_manifest_entry()` only mutates the **in-memory** dict, file by file, inside the loop. `save_manifest()` — the only thing that writes to disk — runs once, *after* the loop finishes. A crash midway through means **nothing** gets persisted: on restart, all 10 files still look unchanged-since-last-save and correctly get reprocessed. Safe, but wasteful — every file gets redone even if 9 of 10 had already succeeded before the crash. The fix is saving the manifest incrementally, after each file, which trades a bit of I/O overhead for not having to redo already-completed work after a crash.

**Q: The manifest tracks content hash only — not which embedding model produced the resulting vectors. If you swap embedding models, does this manifest tell you anything changed?**
A: No — and that's a real blind spot, not a minor one. The manifest would happily report "nothing to ingest" while every existing vector in the store is now incomparable to new queries embedded with the new model. This is the embedding-model-migration failure mode directly — the manifest needs to track the embedding model version alongside the content hash, so a model change is detected as "everything needs re-embedding" even though no source content actually changed.

---

## Lead-Level Judgment (not purely technical)

**Q: You inherit this exact pipeline from a contractor who left. One sprint before a compliance audit. Which gap do you fix first — the manifest's concurrency issue, the near-dup false-positive risk on rate changes, or the single-page OCR miss — and how do you justify that to a non-technical stakeholder?**
A: There's no single correct ranking — what's being evaluated is whether you can connect technical risk to business impact and say it in plain language. A reasonable answer: the near-dup false-positive risk on rate changes is the one most likely to cause a **wrong answer reaching a customer or auditor** (an outdated interest rate silently served as current), which is a compliance and trust problem, not just an engineering one — that goes first. The OCR miss is a *missing* answer, which is bad but more likely to be noticed and escalated. The concurrency issue only manifests under specific operational conditions (simultaneous ingestion jobs) that may not even occur during the audit window. Justifying that priority to a non-technical stakeholder means leading with "which one could give someone false information" before "which one is architecturally messiest" — those are not the same ranking, and conflating them is a common mistake at this level.

## Production Operations — New Documents Arriving Mid-Flight

**Q: A new set of documents arrives while the system is already live, running the email classification project in production. Do you re-initialize/reload everything?**
A: No — and saying "no" without the reasoning behind it would actually be a weak answer. There are two separate systems here, and conflating them is the real trap in this question:
- The **email classifier** (`classifier_core.py`, the v0–v3 prompts) is Claude-based, not fine-tuned. New documents have **zero effect** on it — there's nothing to retrain, because there was never a training step. This is one of RAG's actual selling points, worth saying explicitly: you get to update what the system "knows" without touching the model at all.
- The **knowledge base** (Qdrant / the in-memory store from Chapter 8) is what new documents affect — and even there, the manifest-based incremental ingestion already built means only the **new or changed** files get hashed, chunked, and embedded. Everything already ingested is left untouched. A full reload would mean re-embedding documents that haven't changed at all — wasted compute, wasted time, and on a live system, unnecessary risk for zero benefit.

**Q: Does "new documents arrived" ever mean the classification pipeline itself needs retraining?**
A: Only if the classifier were fine-tuned on a fixed dataset — which it isn't. This is exactly the question that separates someone who understands RAG from someone who's used the term without internalizing why it exists. New product info, new policies, new SOPs go into the **retrieval layer**. The model's weights, whether it's Claude via API or a fine-tuned model, are untouched. If the answer to this question is "yes, retrain it," that's a sign the distinction between "the model's knowledge" and "the system's knowledge" hasn't been made.

**Q: The knowledge base is being actively queried by live traffic while you're updating it with new documents. How do you avoid an inconsistent state?**
A: You don't mutate the live store in place while reads are happening against it. The safer pattern is build-then-swap: ingest and embed the new/changed documents into a separate staging collection, validate it, then atomically point the live retrieval path at the updated version — Qdrant supports this via collection aliases, so the swap is a single pointer change, not a multi-step mutation a query could land in the middle of. Mutating the live collection directly risks a query retrieving a half-updated state — some old chunks, some new, no clean answer either way.

**Q: A new FD Product Guide says the interest rate is 7.5%; the old version, still sitting in the knowledge base, says 7.0%. What happens at query time, and how do you prevent it?**
A: Without an explicit fix, both chunks could get retrieved for the same query, and the model has no principled way to know which one is current — it might pick either, or worse, blend them into a confused answer. This isn't a retrieval-quality problem, it's a **document lifecycle** problem: ingestion needs to actively deprecate the old chunk (remove it or mark it inactive via metadata) when a newer version of the same document type arrives, not just add the new one alongside it. The dedup logic from Chapter 1 catches *identical* content; it does nothing for *superseding* content — that's a different mechanism entirely, and one this pipeline doesn't have yet.

**Q: Is it safe to re-run a failed or partially-completed ingestion job?**
A: Only if the pipeline is idempotent, and right now it isn't guaranteed to be. If documents are inserted into the vector store without a stable, deterministic ID (e.g. a hash of source path + chunk index, rather than an auto-incrementing or random ID), re-running a failed job after a partial success can insert the **same chunks a second time** instead of safely overwriting them. The manifest tells you *which files* need reprocessing; it doesn't, by itself, guarantee that reprocessing a file is a clean no-op if it was already half-written to the vector store before the failure.

---

## Broader Production Issues

**Q: How do you know ingestion is healthy, beyond "the script didn't crash"?**
A: "Didn't crash" is a binary that hides everything that actually matters. Real monitoring means tracking, per run: documents processed vs. failed, OCR confidence distribution (catching the case where OCR "succeeded" but produced garbage), how many documents got flagged as duplicates (a sudden spike might mean someone's re-uploading the same batch by mistake), and total time taken (a slow creep upward is an early warning before it becomes an outage). A pipeline with no metrics can be silently degrading for weeks before anyone notices a customer-facing symptom.

**Q: A bad document gets ingested — wrong file, or a draft instead of the final version. How do you roll back?**
A: Only cleanly if you planned for it. If every ingested chunk's metadata records *which ingestion run* it came from (a run ID or timestamp), rollback is "delete everything tagged with that run ID." Without that, you're left trying to manually identify and remove specific chunks from a vector store after the fact, which is slow and error-prone exactly when you need speed and confidence — right before or during a compliance audit, for instance.

**Q: Who should be allowed to add documents to a production knowledge base feeding a compliance-sensitive system?**
A: This is a process question as much as a technical one, and a Lead-level answer treats it that way. Direct, unreviewed upload into a live, customer-facing knowledge base is a real risk — a draft, a wrong version, or an internal-only document could become something the system actively tells customers. The safer pattern is a staging step: new documents land in a pending state, get reviewed (even a lightweight sign-off), and only then get promoted to the live collection — the same build-then-swap pattern from the live-update question above, now serving an approval gate instead of just a consistency guarantee.

**Q: 10,000 documents arrive at once — a bulk migration, not a steady trickle. What changes about the pipeline?**
A: Everything that was fine as a simple loop now needs to be a queue. Embedding 10,000 documents synchronously, one request at a time, would take a long time and risks hitting rate limits or timing out the whole job on one bad file. The fix is a worker pool processing the queue in parallel, plus per-document failure isolation — one corrupted PDF in a batch of 10,000 should fail *that document* and get logged, not abort the other 9,999.

**Q: The NBFC switches PDF generation tools internally, and the new PDFs have a slightly different internal structure. What breaks, and how do you notice before customers do?**
A: This is exactly the kind of thing that breaks silently. `extract_text()` might return technically-valid-looking text that's actually missing sections, has scrambled column order, or drops a table that used to extract fine. Nothing throws an exception — it just quietly produces worse chunks. The only real defense is the same metric discipline from the monitoring question: track something like average extracted-characters-per-page over time, and alert on a sudden drop, rather than assuming "no errors" means "no problem."

**Q: Six months after going live, you change `chunk_size` or the chunking strategy itself. Do you re-chunk everything already ingested?**
A: This deserves to be asked explicitly, not assumed either way — and the honest answer is "it depends on how much you care about consistency across the knowledge base." Leaving old chunks as-is means the store now has two different chunking strategies coexisting, which can make retrieval quality subtly inconsistent depending on which strategy produced the chunk that happens to get retrieved. Re-chunking everything is the clean answer but is a real cost — full re-embedding of the entire corpus, not just what's new. This is the same category of decision as an embedding model migration: a deliberate, planned re-indexing event, not something that should happen as a side effect of a routine code change nobody flagged as breaking.